# Italy — legal phases, transitions and time in phase

**Intrum CAT instance 4034 · 7 portfolios · seller "Blue SPV" · 48,547 loans · EUR 1,058,690,235.15 original capital**

Measured against the live warehouse on **2026-09-15**. Every figure below comes from a
query in this folder, not from an export.

---

## The answer in one paragraph

Italy has **two phases, not a ladder**: every loan starts *amicable* (`A002`) and 17,003
of 48,547 — **35.024%** — reach *legal* (`L002`). There is no third phase code and no
sub-phase, so there is no decreto ingiuntivo / precetto / pignoramento breakdown to be
had from this feed. Of the loans in legal, **3,234 (19.020%) never escalated at all —
they were bought already in legal**, stamped the day after their batch landed, and they
carry EUR 294.5m, 39.1% of all legal capital. The **13,769 that genuinely escalated took
a median of 198 days** to do it. How long a legal case then *takes* is not answerable:
98.265% of them are still open.

## What is solid and what is shaky

| | |
|---|---|
| **Solid** | The funnel counts. 48,547 loans, 17,003 legal, 31,543 amicable-only — clean, every loan accounted for, no loan in legal without an amicable row. |
| **Solid** | The 198-day median time to escalate, for the 13,769 loans that escalated. Fully observed: both ends are real dates. |
| **Solid** | Escalation rates differ sharply by loan type, and by a lot — mortgages 63.870% against current accounts 34.606%. |
| **Shaky** | "Time in the legal phase." Only 295 loans have both entered legal and closed, median 262 days. Everything else is still running. |
| **Shaky** | Loan type itself. The source field is truncated to 10 characters and the grouping into categories is ours, not Intrum's. |
| **Not available** | Insolvency. Nothing at all — see the coverage check below. |

## Three things that will mislead you if nobody says them

1. **The amicable phase start is a batch-loading date, not a loan event.** `A002` has
   only 8 distinct start dates across 48,546 loans, and each one's loan count is exactly
   one portfolio's size. Everyone in a batch is stamped the day the batch arrived.
2. **`PRODUCT_ID`, `IS_TITLED` and `CASE_PHASE` are the legal status wearing three
   different hats.** `product_id=7` and `case_phase=2000` split the book identically.
   "Titled loans escalate more" is not a finding — it is the same column twice. The one
   genuinely independent attribute is `IS_SECURED`.
3. **A quarter of the rows are not loans.** 13,020 rows are `SPESE` / `SPESE LEGALI` —
   expense lines, averaging EUR 723 against EUR 86,805 for a mortgage. They are 26.818%
   of the row count and 0.889% of the capital. Any per-loan average that leaves them in
   is roughly half of the truth.

## On the status-code mapping you supplied

The 103-code Intrum lookup is in `main_status_code.csv` and it does decode this book
perfectly — but it decodes it to **two codes**: `64` *On-going* and `6` *Completed*, i.e.
open and closed. It does **not** yield an amicable/legal split for Italy, and the split
above comes from `GROUP_CASE_PHASE` instead. (`CASES` and `CASE_STATUS_CHANGE_HISTORY`
also disagree about the open code — 0 vs 64 — which is checked below.)

## Where the numbers come from

| file | what it gives you |
|---|---|
| `loan_phase_history.sql` | one row per loan — phase dates, durations, loan type. **The authority: every guard and caveat is in its header.** |
| `phase_summary.sql` | the funnel and durations aggregated by loan type and route |
| `cat_table_coverage.sql` | proof of which CAT tables are empty for this instance |
| `main_status_code.csv` | Intrum's own 103-code lookup, with their comments |


In [ ]:
import sys
sys.path.insert(0, "../../packages")   # notebook sits in work/italy/legal/
import datalake
import pandas as pd

# Money and ratios to 5 dp, per the workspace convention. This changes only what
# is PRINTED — it never alters the values, so nothing is rounded away.
pd.set_option("display.float_format", lambda v: f"{v:,.5f}")
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 120)

EXPORT_NAME = "italy_legal"


## 1 — Load the loan-level dataset

`convert=False` keeps Snowflake's `Decimal` values instead of float64, which is the
workspace rule for anything you sum or reconcile. Dates are coerced explicitly
afterwards so the duration columns stay usable.

In [ ]:
# ~2 minutes. Reads all nine CAT extracts for instance 4034 and the phase feed.
loans = datalake.q("work/italy/legal/loan_phase_history", convert=False)

for c in ["REGISTRATION_DATE", "DEFAULT_DATE", "CLOSING_DATE",
          "AMICABLE_START", "LEGAL_START", "LEGAL_LAST_STAMP", "OBSERVATION_END"]:
    loans[c] = pd.to_datetime(loans[c])

# Durations arrive as Decimal; make them numeric so describe()/median() work.
for c in ["DAYS_IN_AMICABLE", "DAYS_IN_LEGAL", "LEGAL_STAMP_COUNT"]:
    loans[c] = pd.to_numeric(loans[c])

print(f"{len(loans):,} rows x {len(loans.columns)} columns")
print(f"observation window ends {loans.OBSERVATION_END.max():%Y-%m-%d}")
loans.head(3)


In [ ]:
# --- checks worth running before trusting anything below ---------------------
# Each of these encodes a claim made in the markdown above.
assert loans.LOAN_ID.is_unique, "must be one row per loan"
assert len(loans) == 48_547, f"expected 48,547 loans, got {len(loans):,}"

# Every loan in legal also has an amicable row: the funnel has no side entrance.
assert not (loans.REACHED_LEGAL & loans.AMICABLE_START.isna()).any(), \
    "a loan reached L002 without an A002 row — the funnel is not a funnel"

# The day-1 spike is a clean gap, not a threshold we tuned: days 2-10 are
# empty, so 'Arrived in legal' cannot be an off-by-one.
gap = loans.loc[loans.REACHED_LEGAL, "DAYS_IN_AMICABLE"]
assert not gap.between(2, 10).any(), "the day-1 spike is no longer cleanly separated"
assert (gap == 1).sum() == 3_234, "the size of the day-1 spike has changed"

# PRODUCT_ID is the legal marker, not a loan type. If this ever fails, the
# warning in the markdown is out of date and product_id became meaningful.
xtab = pd.crosstab(loans.PRODUCT_ID, loans.CURRENT_CASE_PHASE)
print("PRODUCT_ID vs CURRENT_CASE_PHASE — should be near-diagonal:")
print(xtab)


## 2 — The funnel

One transition exists: `A002` → `L002`. This is the loan count you asked for, at each
stage and for the move between them.

In [ ]:
n = len(loans)
funnel = pd.DataFrame([
    {"stage": "1. Amicable (A002) — every loan",
     "n_loans": int(loans.AMICABLE_START.notna().sum()),
     "capital_eur": float(loans.loc[loans.AMICABLE_START.notna(), "ORIGINAL_CAPITAL"].sum())},
    {"stage": "2. Legal (L002)",
     "n_loans": int(loans.REACHED_LEGAL.sum()),
     "capital_eur": float(loans.loc[loans.REACHED_LEGAL, "ORIGINAL_CAPITAL"].sum())},
    {"stage": "   ...of which ARRIVED in legal (bought that way)",
     "n_loans": int((loans.ESCALATION_ROUTE == "Arrived in legal").sum()),
     "capital_eur": float(loans.loc[loans.ESCALATION_ROUTE == "Arrived in legal", "ORIGINAL_CAPITAL"].sum())},
    {"stage": "   ...of which ESCALATED (a real A->L transition)",
     "n_loans": int((loans.ESCALATION_ROUTE == "Escalated to legal").sum()),
     "capital_eur": float(loans.loc[loans.ESCALATION_ROUTE == "Escalated to legal", "ORIGINAL_CAPITAL"].sum())},
    {"stage": "Never left amicable",
     "n_loans": int((loans.ESCALATION_ROUTE == "Never legal").sum()),
     "capital_eur": float(loans.loc[loans.ESCALATION_ROUTE == "Never legal", "ORIGINAL_CAPITAL"].sum())},
])
funnel["pct_of_loans"] = funnel.n_loans / n
funnel


**Read it this way.** Of roughly 48,500 Italian loans, about 17,000 are in legal
proceedings. But only 13,769 of those were *sent* there by the servicer — the other 3,234
were already in court when Brocc bought them, and those 3,234 are the expensive ones:
EUR 294.5m of the EUR 752.6m sitting in legal.

## 3 — How long loans spend in each phase

Two different questions, and only one of them has a real answer.

In [ ]:
esc = loans[loans.ESCALATION_ROUTE == "Escalated to legal"]

print("TIME TO ESCALATE — amicable onboarding to first legal action")
print("Fully observed: both ends are real dates. This number is solid.\n")
print(esc.DAYS_IN_AMICABLE.describe(percentiles=[.25, .5, .75, .9]).to_string())


In [ ]:
legal = loans[loans.REACHED_LEGAL]
closed = legal[~legal.LEGAL_IS_CENSORED]

print("TIME IN LEGAL")
print(f"  still open (censored) : {legal.LEGAL_IS_CENSORED.sum():,} "
      f"({legal.LEGAL_IS_CENSORED.mean():.5%})")
print(f"  closed               : {len(closed):,}\n")
print("⚠ The censored figure below is 'how long ago they entered', NOT 'how long a")
print("  legal case takes'. It is bounded by the feed horizon, not by an outcome.")
print(f"  median days in legal, ALL legal loans   : {legal.DAYS_IN_LEGAL.median():,.0f}")
print()
print("The only uncensored estimate — loans that entered legal AND closed:")
print(closed.DAYS_IN_LEGAL.describe(percentiles=[.25, .5, .75, .9]).to_string())
print()
print("⚠ That rests on 295 loans out of 17,003. Quote it as indicative, with the n.")


## 4 — Broken down by loan type

`LOAN_TYPE_RAW` is Intrum's own text and is quotable. `LOAN_TYPE_GROUP` is **ours** — a
reading of that text, defined in one `case` expression in `loan_phase_history.sql`. Some
calls are judgement, and the source text is truncated to 10 characters, so treat the
groups as a working taxonomy rather than a blessed one.

In [ ]:
by_type = loans.groupby("LOAN_TYPE_GROUP").agg(
    n_loans=("LOAN_ID", "size"),
    capital_eur=("ORIGINAL_CAPITAL", "sum"),
    n_legal=("REACHED_LEGAL", "sum"),
)
by_type["pct_of_loans"]   = by_type.n_loans / len(loans)
by_type["escalation_rate"] = by_type.n_legal / by_type.n_loans
by_type["avg_capital_eur"] = by_type.capital_eur.astype(float) / by_type.n_loans

# Median days to escalate, counting ONLY genuine escalations — including the
# 'arrived in legal' loans would drag every median toward 1 day.
by_type["median_days_to_escalate"] = esc.groupby("LOAN_TYPE_GROUP").DAYS_IN_AMICABLE.median()

by_type.sort_values("n_loans", ascending=False)


**The pattern, in words.** Secured-ish, big-ticket credit goes to court and goes there
fast; small unsecured balances mostly do not. Leasing escalates at 75.4%, mortgages at
63.9%, guarantees at 59.7% — against 34.6% for current-account overdrafts, which are
41% of the book by count. Mortgages are the clearest case: **3,504 mortgages, of which
2,238 end up in legal, carrying EUR 303.9m — the single biggest block of legal capital
on the book.**

A caution for the PM version of this: mortgages show a *median 1 day* to escalate if you
do not split the routes, because 1,139 of them arrived in legal already. Split out, the
mortgages that genuinely escalated took a median of 248 days — the **slowest** of any
major type, not the fastest.

In [ ]:
# Secured is the one attribute that is genuinely independent of the phase
# (unlike IS_TITLED / PRODUCT_ID / CASE_PHASE — see the check in section 1).
sec = loans.groupby("IS_SECURED").agg(
    n_loans=("LOAN_ID", "size"),
    capital_eur=("ORIGINAL_CAPITAL", "sum"),
    n_legal=("REACHED_LEGAL", "sum"))
sec["escalation_rate"] = sec.n_legal / sec.n_loans
print(sec.to_string())
print("\n⚠ Only 404 loans are flagged secured — 0.832% of the book. The 75% escalation")
print("  rate is a real contrast but rests on a small, possibly under-flagged group.")


## 5 — By portfolio, with the exposure caveat

Escalation rates are not comparable across portfolios without their onboarding dates.
A batch loaded in December 2025 has had eight months to escalate; one loaded in July 2024
has had two years.

In [ ]:
by_pf = loans.groupby("PORTFOLIO_ID").agg(
    onboarded=("AMICABLE_START", "min"),
    n_loans=("LOAN_ID", "size"),
    capital_eur=("ORIGINAL_CAPITAL", "sum"),
    n_legal=("REACHED_LEGAL", "sum"),
    obs_end=("OBSERVATION_END", "max"))
by_pf["escalation_rate"] = by_pf.n_legal / by_pf.n_loans
by_pf["days_observed"]   = (by_pf.obs_end - by_pf.onboarded).dt.days
by_pf.drop(columns="obs_end").sort_values("onboarded")


⚠ **`IT25014A010000` shows an 88.710% escalation rate on 186 loans, but it was onboarded
2025-12-07 and has only 236 days of history.** That is not a portfolio that escalates
faster — it is a small, mostly-already-in-legal batch. Do not put it in a league table
next to the 2024 books.

## 6 — The status codes, decoded

Your 103-code lookup, applied. It resolves everything — and that is exactly the point:
what it resolves to is *open* and *closed*.

In [ ]:
codes = pd.read_csv("main_status_code.csv", dtype={"CODE": str})
print(f"{len(codes)} codes in Intrum's lookup\n")

observed = (loans.assign(MAIN_STATUS_CODE=loans.MAIN_STATUS_CODE.astype(str))
                 .groupby("MAIN_STATUS_CODE")
                 .agg(n_loans=("LOAN_ID", "size"),
                      capital_eur=("ORIGINAL_CAPITAL", "sum"))
                 .reset_index()
                 .merge(codes, left_on="MAIN_STATUS_CODE", right_on="CODE", how="left"))
observed["pct_of_loans"] = observed.n_loans / len(loans)
print("From CASES (this is also what PROD.DIM_CASE carries):")
print(observed[["MAIN_STATUS_CODE", "DESCRIPTION", "n_loans", "pct_of_loans"]].to_string(index=False))
print()
print("⚠ Two codes only, and both CAT tables do not agree on the OPEN one:")
print("    CASES                      says 0  'Registered'  (46,474 loans)")
print("    CASE_STATUS_CHANGE_HISTORY says 64 'On-going'    (48,546 loans)")
print("  'Registered — no action taken yet' is plainly wrong for a book that is 35%")
print("  in legal, so read the 0 as an unset default. Worth raising with the data team.")
print("  Either way the dictionary gives no amicable/legal split for Italy.")


## 7 — Insolvency and the other legal fields: what is actually there

You asked what else CAT holds on insolvency and legal matters. The honest answer for
Italy is *nothing* — and it is worth seeing rather than taking on trust, because an
empty table and a forgotten join look identical in a result set.

In [ ]:
# ~4 minutes. Counts every CAT table for instance 4034.
coverage = datalake.q("work/italy/legal/cat_table_coverage")
coverage


**What that costs us.** These tables are not sparse for Italy — they are absent:

- `BANKRUPTCY_RECORD` would have given `INSOLVENCY_OPEN_DATE`, `BANKRUPTCY_END_DATE`,
  `AFFIDAVIT_DATE` and `ARREST_WARRANT_DATE`. **0 rows.** So there is no insolvency
  measure of any kind on this book — not a count, not a duration.
- `LEGAL_STEPS` carries `LEGAL_STEP` with a `START_DATE` *and* an `END_DATE`. That is
  precisely the per-stage duration table you were hoping for, and it is the reason the
  answer to "which legal phases are there" is only two. **0 rows.**
- `LEGAL_INFO` would have given judgement and summons dates and verdict amounts. **0 rows.**
- `ASSIGNMENT` would have shown the external lawyer or bailiff. **0 rows.**

The same fields on `CASES` are 0% filled too — `COURT_PROCEDURE`, `JUDGEMENT_DATE`,
`SERVING_DATE`, `LEGAL_CLAIM_REASON`, `STATUTE_BARRED_DATE`, `DEBT_ENFORCEMENT_NUMBER`,
`DENOUNCEMENT_DATE`.

**This is a question for Intrum, not a query to write.** Ask whether the Italian servicer
populates the legal module at all, or whether those tables exist for 4034 and are simply
not in the extract we receive. Until one of those is true, phase-level legal analysis for
Italy stops at amicable-vs-legal.

## 8 — On CAT-vs-PROD for this book

Worth recording, because the premise for reaching for CAT does not hold here. PROD's
loan-level collections have a hard floor at 2014-10-10, which is why pre-2014 books have
to be read from raw CAT. **Instance 4034 is a 2024+ book** — registrations run 2024-07-21
to 2025-12-07 — so there is no extra history in CAT to recover. PROD holds the same
48,547 cases and agrees exactly on the status split (46,474 / 2,073).

CAT is still the right source here, but for a different reason: **the phase data only
exists in raw.** `GROUP_CASE_PHASE` is not modelled in PROD and is not even declared as a
dbt source. `PROD.DIM_CASE.STATUS_DESCRIPTION` is NULL for every one of these loans, so
PROD cannot decode the status either.

One genuine piece of history does reach back: `DEFAULT_DATE` runs from **1993-07-30**, so
the underlying debts are old even though Intrum's case records are not.

## 9 — The aggregated summary table

Same numbers as above, computed server-side, if you want one table to hand over.

In [ ]:
summary = datalake.q("work/italy/legal/phase_summary", convert=False)
summary


In [ ]:
# --- the payment funnel: amicable -> legal ------------------------------------
# Pairs loan_phase_history (phases) with loan_payments (money + customers).
# Both are one row per loan, same 48,547 loans, same guards.
#
# ⚠ NOTHING HERE IS CENSORED, which is the only reason the durations mean
#   anything. Measured 2026-09-15: registration, the first A002 spell and the
#   first transaction ALL begin 2024-07-21, so every clock has a real start.
#   On Portugal the same query would be meaningless -- the case history there
#   starts nine years after acquisition.
pay = datalake.q("work/italy/legal/loan_payments", convert=False)

for c in ["FIRST_PAYMENT_DATE", "LAST_PAYMENT_DATE", "FIRST_PAYMENT_AFTER_LEGAL"]:
    pay[c] = pd.to_datetime(pay[c])
for c in ["N_DEBTORS", "N_PAYMENTS", "N_PAYMENTS_AFTER_LEGAL"]:
    pay[c] = pd.to_numeric(pay[c])
pay["TOTAL_COLLECTIONS"] = pay["TOTAL_COLLECTIONS"].astype(float)

f = loans.merge(pay, on="LOAN_ID", how="left", validate="one_to_one")
assert len(f) == len(loans), "the payment join changed the row count"

# A payment counts as "in amicable" only if it lands BEFORE the loan's own L002
# stamp. Without that test, a loan that paid solely in legal would be credited
# to both stages and the funnel would not be a funnel.
f["PAID_IN_AMICABLE"] = f.FIRST_PAYMENT_DATE.notna() & (
    f.LEGAL_START.isna() | (f.FIRST_PAYMENT_DATE < f.LEGAL_START))
f["DAYS_TO_PAY_AMICABLE"] = (
    (f.FIRST_PAYMENT_DATE - f.AMICABLE_START).dt.days).where(f.PAID_IN_AMICABLE)

# The legal clock starts at the loan's OWN L002 stamp, so a loan bought already
# in legal is measured from the day it was stamped, not from a servicing
# decision. That is why 2a and 2b are split out below.
f["PAID_IN_LEGAL"] = f.FIRST_PAYMENT_AFTER_LEGAL.notna()
f["DAYS_TO_PAY_LEGAL"] = (f.FIRST_PAYMENT_AFTER_LEGAL - f.LEGAL_START).dt.days


def stage(label, mask, paid_col, days_col):
    """One funnel row. Every metric is computed on the same loan subset."""
    d = f.loc[mask]
    paid = d[paid_col]
    return {
        "stage":            label,
        "n_loans":          int(len(d)),
        # ⚠ DISTINCT, never a sum: 20,567 customers hold 48,546 loans on this
        # book (2.36 each), so summing a per-loan count overstates by ~2.36x.
        "n_customers":      int(d.DEBTOR_ID.nunique()),
        "capital_eur":      float(d.ORIGINAL_CAPITAL.sum()),
        "n_made_payment":   int(paid.sum()),
        "pct_made_payment": float(paid.mean()) if len(d) else float("nan"),
        "median_days_to_first_payment": float(d.loc[paid, days_col].median()),
        # ⚠ LIFETIME collections on these loans, not collections in this phase.
        # The feed has no per-phase split; do not read row 2 as "legal cash".
        "collections_eur_lifetime": float(d.TOTAL_COLLECTIONS.sum()),
    }


funnel_payments = pd.DataFrame([
    stage("1. Amicable (A002) - every loan",
          f.AMICABLE_START.notna(), "PAID_IN_AMICABLE", "DAYS_TO_PAY_AMICABLE"),
    stage("2. Legal (L002) - all",
          f.REACHED_LEGAL, "PAID_IN_LEGAL", "DAYS_TO_PAY_LEGAL"),
    stage("   2a. ...ESCALATED from amicable",
          f.ESCALATION_ROUTE == "Escalated to legal",
          "PAID_IN_LEGAL", "DAYS_TO_PAY_LEGAL"),
    stage("   2b. ...ARRIVED already in legal",
          f.ESCALATION_ROUTE == "Arrived in legal",
          "PAID_IN_LEGAL", "DAYS_TO_PAY_LEGAL"),
])

print("PAYMENT FUNNEL - loans, customers, and days to the first payment")
print("Amicable clock starts at A002. Legal clock starts at the loan's own L002")
print("stamp, so 2a is a servicing decision and 2b is the day it was bought.\n")
print(funnel_payments.to_string(index=False))
funnel_payments


## 10 — Export

In [ ]:
for name, df in (("loan_phase_history", loans), ("phase_summary", summary)):
    path = f"{EXPORT_NAME}_{name}.csv"
    df.to_csv(path, index=False)
    print(f"{len(df):>9,} rows -> {path}")
